# Notebook 03: Tool Integration and OpenAI Function Calling

Welcome to the exciting world of AI agents! In this notebook, you'll learn how to integrate Large Language Models (LLMs) with tools to create agents that can take actions.

## Learning Objectives

By the end of this notebook, you will be able to:

1. Integrate OpenAI's GPT models with LangGraph
2. Define and bind tools to LLMs
3. Implement function/tool calling patterns
4. Manage message state with the `add_messages` reducer
5. Build AI agents that can use multiple tools
6. Handle tool execution and errors gracefully

## Prerequisites

- Completed Notebooks 01-02
- OpenAI API key (get one at https://platform.openai.com/api-keys)
- `.env` file configured with `OPENAI_API_KEY`

## Estimated Time

120-150 minutes

## Complexity Level

Intermediate (3/5)

## LangGraph Version

This notebook targets **LangGraph 1.1.9**.

## 1. Introduction: What are AI Agents?

### From Static to Dynamic

So far, we've built workflows where all logic is hardcoded:
- Predefined functions
- Fixed routing rules
- Static processing steps

**AI Agents** are different. They can:
- **Reason**: Analyze situations and make decisions
- **Use Tools**: Execute functions to interact with the world
- **Adapt**: Change behavior based on context
- **Learn**: Improve through feedback

### The Tool Calling Pattern

Modern LLMs like GPT-4 support **function/tool calling**:

1. You define available tools (functions the LLM can call)
2. The LLM receives a user query
3. The LLM decides which tool(s) to use
4. Your code executes the tool
5. Results are sent back to the LLM
6. The LLM generates a final response

### Example Flow

```
User: "What's the weather in Paris?"
  ↓
LLM: "I need to call get_weather(city='Paris')"
  ↓
Tool Execution: get_weather('Paris') → "Sunny, 22°C"
  ↓
LLM: "The weather in Paris is sunny with a temperature of 22°C."
```

## 2. Setup

### Install Required Packages

Make sure you have the required packages installed (should be in requirements.txt)

In [ ]:
# Uncomment if you need to install:
# !pip install langgraph langchain-openai langchain-core python-dotenv

### Import Libraries and Load API Key

In [1]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check API key
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found! Please:\n"
        "1. Create a .env file in the project root\n"
        "2. Add: OPENAI_API_KEY=your_key_here\n"
        "3. Get a key from https://platform.openai.com/api-keys"
    )

print("✅ Imports successful!")
print("🤖 Ready to build AI agents!")

✅ Imports successful!
🤖 Ready to build AI agents!


In [2]:
# Verify LangGraph version
import langgraph
#print(f"LangGraph version: {langgraph.__version__}")
# Should show 1.1.9+

## 3. Understanding Message State

Before building agents, let's understand how messages work in LangGraph.

### The `add_messages` Reducer

Unlike previous notebooks where state updates **replaced** values, agent state needs to **accumulate** messages:

In [3]:
# State for agents - note the Annotated type with add_messages
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # Messages accumulate instead of replacing

# Let's test how add_messages works
from langgraph.graph.message import add_messages as add_msg_fn

# Existing messages
existing = [HumanMessage(content="Hello")]

# New messages
new = [AIMessage(content="Hi there!")]

# add_messages combines them
result = add_msg_fn(existing, new)

print("Existing messages:", len(existing))
print("New messages:", len(new))
print("Combined messages:", len(result))
print("\nMessages:")
for msg in result:
    print(f"  {type(msg).__name__}: {msg.content}")

Existing messages: 1
New messages: 1
Combined messages: 2

Messages:
  HumanMessage: Hello
  AIMessage: Hi there!


**Key Point**: `Annotated[list, add_messages]` tells LangGraph to **append** new messages instead of replacing the entire list.

## 🆕 Built-in MessagesState

Instead of defining message state manually, LangGraph 1.1.9 provides `MessagesState`:

```python
# New: use the built-in MessagesState
from langgraph.graph import MessagesState

# Old way (still valid):
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
```

Both approaches work identically. `MessagesState` is just a convenience.

In [4]:
# Option A: Built-in MessagesState (recommended for message-based agents)
from langgraph.graph import MessagesState

# Option B: Manual definition (equivalent, more explicit)
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# Both are equivalent — we'll use MessagesState going forward

## 4. Example 1: Simple Weather Agent

Let's build our first tool-using agent!

### Step 1: Define Tools

Tools are Python functions decorated with `@tool`:

In [5]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city.
    
    Args:
        city: The name of the city to get weather for.
        
    Returns:
        A string describing the current weather.
    """
    # Simulated weather data
    weather_data = {
        "paris": "Sunny, 22°C",
        "london": "Cloudy, 15°C",
        "tokyo": "Rainy, 18°C",
        "new york": "Partly cloudy, 20°C",
    }
    
    city_lower = city.lower()
    if city_lower in weather_data:
        return f"The weather in {city} is {weather_data[city_lower]}"
    else:
        return f"Sorry, I don't have weather data for {city}"

# Test the tool directly
print("Testing tool:")
print(get_weather.invoke({"city": "Paris"}))

Testing tool:
The weather in Paris is Sunny, 22°C


**Important Notes**:
- The `@tool` decorator converts a function into a LangChain tool
- The docstring is crucial - the LLM uses it to understand what the tool does
- Type hints help the LLM understand parameter types

### Step 2: Create LLM with Tools

Now we bind the tool to the LLM:

In [6]:
# Create the LLM
llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)

# Bind tools to the LLM
tools = [get_weather]
llm_with_tools = llm.bind_tools(tools)

print("✅ LLM configured with tools")

✅ LLM configured with tools


### Step 3: Build the Agent Graph

An agent graph typically has this pattern:

```
START → agent → should_continue?
                  ├─ "continue" → tools → agent (loop)
                  └─ "end" → END
```

## ⚠️ Deprecation Notice: create_react_agent

`create_react_agent` from `langgraph.prebuilt` is **deprecated** in LangGraph 1.0+ and will be removed in LangGraph 2.0. It still works in 1.1.9 but shows a deprecation warning.

The new approach uses `create_agent` from `langchain.agents`:

```python
# ❌ DEPRECATED — avoid in new code
from langgraph.prebuilt import create_react_agent
app = create_react_agent(llm, tools=tools)

# ✅ NEW — recommended approach
from langchain.agents import create_agent
app = create_agent(llm, tools=tools, system_prompt="You are a helpful assistant.")
```

Key differences in `create_agent`:
- `system_prompt` parameter replaces `prompt`/`state_modifier`  
- Supports middleware (retry, content moderation) via `middleware=` parameter
- `AgentState` now lives in `langchain.agents.AgentState`

**Note**: The manual agent loop pattern taught in this notebook (building the graph yourself with `call_model` and `execute_tools` nodes) is NOT deprecated and gives you full control. `create_agent` is a convenience wrapper around this same pattern.

In [7]:
from typing import Literal

# State for our weather agent
class WeatherAgentState(TypedDict):
    messages: Annotated[list, add_messages]

# Node 1: Call the LLM
def call_model(state: WeatherAgentState) -> dict:
    """Call the LLM with the current messages."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Node 2: Execute tools
def call_tools(state: WeatherAgentState) -> dict:
    """Execute the tools requested by the LLM."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # Get tool calls from the last message
    tool_calls = last_message.tool_calls
    
    # Execute each tool
    tool_messages = []
    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"🔧 Calling tool: {tool_name}({tool_args})")
        
        # Find and execute the tool
        tool = {t.name: t for t in tools}[tool_name]
        result = tool.invoke(tool_args)
        
        print(f"📊 Tool result: {result}")
        
        # Create a tool message with the result
        tool_messages.append(
            ToolMessage(
                content=result,
                tool_call_id=tool_call["id"]
            )
        )
    
    return {"messages": tool_messages}

# Routing function: Should we continue or end?
def should_continue(state: WeatherAgentState) -> Literal["tools", "__end__"]:
    """Determine if we should call tools or end."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # If the LLM makes tool calls, route to tools
    if last_message.tool_calls:
        print("🔀 LLM requested tool calls, routing to tools")
        return "tools"
    # Otherwise, end
    print("🏁 No more tool calls, ending")
    return "__end__"

In [8]:
# Build the graph
weather_graph = StateGraph(WeatherAgentState)

# Add nodes
weather_graph.add_node("agent", call_model)
weather_graph.add_node("tools", call_tools)

# Add edges
weather_graph.add_edge(START, "agent")

# Conditional edge from agent
weather_graph.add_conditional_edges(
    "agent",
    should_continue,
)

# Tools always go back to agent
weather_graph.add_edge("tools", "agent")

# Compile
weather_agent = weather_graph.compile()

print("✅ Weather agent ready!")

✅ Weather agent ready!


### Step 4: Test the Agent

In [9]:
# Ask about weather
query = "What's the weather like in Paris?"

print("=" * 60)
print(f"User: {query}")
print("=" * 60)

result = weather_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

# Print final response
final_response = result["messages"][-1].content
print("\n" + "=" * 60)
print(f"Agent: {final_response}")
print("=" * 60)

User: What's the weather like in Paris?
🔀 LLM requested tool calls, routing to tools
🔧 Calling tool: get_weather({'city': 'Paris'})
📊 Tool result: The weather in Paris is Sunny, 22°C
🏁 No more tool calls, ending

Agent: The current weather in Paris is sunny with a temperature of 22°C.


In [10]:
# Try a query that doesn't need tools
query2 = "What is the capital of France?"

print("\n" + "=" * 60)
print(f"User: {query2}")
print("=" * 60)

result2 = weather_agent.invoke({
    "messages": [HumanMessage(content=query2)]
})

final_response2 = result2["messages"][-1].content
print("\n" + "=" * 60)
print(f"Agent: {final_response2}")
print("=" * 60)


User: What is the capital of France?
🏁 No more tool calls, ending

Agent: The capital of France is Paris.


## Parallel Tool Calls

Modern LLMs (GPT-4o, Claude) can call multiple tools simultaneously in a single response. LangGraph handles this automatically — the `AIMessage` may contain multiple `tool_calls`, and `ToolNode` executes them all.

In [11]:
# Example: LLM decides to call multiple tools in parallel
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# Simulating what GPT-4o returns when it wants to call tools in parallel:
parallel_response = AIMessage(
    content="",
    tool_calls=[
        {"name": "get_weather", "args": {"city": "Paris"}, "id": "call_1"},
        {"name": "get_weather", "args": {"city": "London"}, "id": "call_2"},
        {"name": "get_weather", "args": {"city": "Berlin"}, "id": "call_3"},
    ]
)
print("LLM requested parallel tool calls:")
for tc in parallel_response.tool_calls:
    print(f"  → {tc['name']}({tc['args']})")

# ToolNode handles all of them automatically
print("\nWith ToolNode, all 3 calls execute and return results before the next LLM call.")
print("This is a major efficiency gain vs sequential tool execution.")

LLM requested parallel tool calls:
  → get_weather({'city': 'Paris'})
  → get_weather({'city': 'London'})
  → get_weather({'city': 'Berlin'})

With ToolNode, all 3 calls execute and return results before the next LLM call.
This is a major efficiency gain vs sequential tool execution.


In [12]:
# ============================================================
# REAL PREBUILT TOOLNODE DEMO
# Free APIs used:
#   • Open-Meteo   → weather  (no key needed)
#   • REST Countries → country info (no key needed)
# ============================================================


import requests
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# ── Tool 1: Real Weather via Open-Meteo (100% free, no key) ──
@tool
def get_weather(city: str) -> str:
    """Get the current real weather for a given city using Open-Meteo API."""
    try:
        # Step 1: Geocode city → lat/lon
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"
        geo_resp = requests.get(geo_url, params={"name": city, "count": 1}).json()

        if not geo_resp.get("results"):
            return f"Could not find city: {city}"

        loc  = geo_resp["results"][0]
        lat  = loc["latitude"]
        lon  = loc["longitude"]
        name = loc["name"]
        country = loc.get("country", "")

        # Step 2: Fetch current weather
        wx_url  = "https://api.open-meteo.com/v1/forecast"
        wx_resp = requests.get(wx_url, params={
            "latitude":        lat,
            "longitude":       lon,
            "current_weather": True,
            "wind_speed_unit": "mph",
        }).json()

        cw   = wx_resp["current_weather"]
        temp = cw["temperature"]
        wind = cw["windspeed"]
        code = cw["weathercode"]

        # WMO weather code → human label
        weather_labels = {
            0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy",
            3: "Overcast", 45: "Foggy", 48: "Icy fog",
            51: "Light drizzle", 61: "Slight rain", 63: "Moderate rain",
            65: "Heavy rain", 71: "Slight snow", 73: "Moderate snow",
            80: "Slight showers", 81: "Moderate showers", 95: "Thunderstorm",
        }
        condition = weather_labels.get(code, f"Code {code}")

        return (
            f"🌍 {name}, {country}\n"
            f"🌡  Temperature : {temp}°C\n"
            f"💨 Wind speed  : {wind} mph\n"
            f"☁  Condition   : {condition}"
        )

    except Exception as e:
        return f"Weather fetch failed for {city}: {e}"


# ── Tool 2: Country Info via REST Countries (100% free, no key) ──
@tool
def get_country_info(country: str) -> str:
    """Get detailed information about a country: capital, population, currency, languages."""
    try:
        url  = f"https://restcountries.com/v3.1/name/{country}"
        resp = requests.get(url).json()

        if isinstance(resp, dict) and resp.get("status") == 404:
            return f"Country not found: {country}"

        data       = resp[0]
        name       = data["name"]["common"]
        capital    = data.get("capital", ["N/A"])[0]
        population = f"{data.get('population', 0):,}"
        region     = data.get("region", "N/A")
        subregion  = data.get("subregion", "N/A")

        currencies = ", ".join(
            f"{v['name']} ({v.get('symbol', '?')})"
            for v in data.get("currencies", {}).values()
        )
        languages  = ", ".join(data.get("languages", {}).values())
        timezones  = ", ".join(data.get("timezones", [])[:2])

        return (
            f"🌐 {name}\n"
            f"🏛  Capital    : {capital}\n"
            f"👥 Population : {population}\n"
            f"📍 Region     : {subregion}, {region}\n"
            f"💰 Currency   : {currencies}\n"
            f"🗣  Languages  : {languages}\n"
            f"🕐 Timezones  : {timezones}"
        )

    except Exception as e:
        return f"Country info fetch failed for {country}: {e}"


# ── Tool 3: Currency Exchange via Frankfurter (100% free, no key) ──
@tool
def get_exchange_rate(base: str, target: str) -> str:
    """Get the current exchange rate between two currencies. E.g. base='USD', target='EUR'."""
    try:
        url  = f"https://api.frankfurter.app/latest?from={base.upper()}&to={target.upper()}"
        resp = requests.get(url).json()

        if "error" in resp:
            return f"Could not get rate for {base} → {target}: {resp['error']}"

        rate = resp["rates"].get(target.upper())
        date = resp.get("date", "today")

        return (
            f"💱 Exchange Rate ({date})\n"
            f"   1 {base.upper()} = {rate} {target.upper()}"
        )

    except Exception as e:
        return f"Exchange rate fetch failed: {e}"



tools = [get_weather, get_country_info, get_exchange_rate]
llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

def llm_node(state: MessagesState):
    return {"messages": [llm.invoke(state["messages"])]}

tool_node = ToolNode(tools)   # ← handles ALL tool calls automatically

builder = StateGraph(MessagesState)
builder.add_node("llm",   llm_node)
builder.add_node("tools", tool_node)
builder.add_edge(START,   "llm")

# tools_condition: if LLM returned tool_calls → go to tools, else → END
builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")   # after tools run, LLM sees results and responds

graph = builder.compile()

# ── Test Queries ───────────────────────────────────────────────
queries = [
    # Tests parallel tool calls (LLM calls get_weather 3x simultaneously)
    "What is the current weather in Paris, Tokyo, and Sydney?",

    # Tests mixed tool calls
    "Tell me about France — its capital, population, and weather right now.",

    # Tests all 3 tools
    "Compare weather in New York vs London, and what's the USD to GBP exchange rate?",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"❓ Query: {query}")
    print(f"{'='*60}")

    result = graph.invoke({"messages": [HumanMessage(content=query)]})
    final  = result["messages"][-1].content
    print(f"\n🤖 Answer:\n{final}")


❓ Query: What is the current weather in Paris, Tokyo, and Sydney?

🤖 Answer:
Here is the current weather for the cities you requested:

### Paris, France
- 🌡 Temperature: 11.7°C
- 💨 Wind Speed: 3.6 mph
- ☁ Condition: Mainly clear

### Tokyo, Japan
- 🌡 Temperature: 19.7°C
- 💨 Wind Speed: 7.3 mph
- ☁ Condition: Clear sky

### Sydney, Australia
- 🌡 Temperature: 20.1°C
- 💨 Wind Speed: 7.1 mph
- ☁ Condition: Clear sky

❓ Query: Tell me about France — its capital, population, and weather right now.

🤖 Answer:
Here’s some information about France:

- **Capital**: Paris
- **Population**: Approximately 66,351,959
- **Currency**: Euro (€)
- **Languages**: French

### Current Weather in Paris:
- **Temperature**: 11.7°C
- **Wind Speed**: 3.6 mph
- **Condition**: Mainly clear

If you need more information, feel free to ask!

❓ Query: Compare weather in New York vs London, and what's the USD to GBP exchange rate?

🤖 Answer:
Here's the current weather comparison between New York and London, along wi

## 5. Example 2: Math Assistant with Calculator

Let's build a more sophisticated agent with multiple math tools.

In [13]:
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together.
    
    Args:
        a: First number
        b: Second number
        
    Returns:
        The sum of a and b
    """
    result = a + b
    print(f"  ➕ {a} + {b} = {result}")
    return result

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together.
    
    Args:
        a: First number
        b: Second number
        
    Returns:
        The product of a and b
    """
    result = a * b
    print(f"  ✖️  {a} × {b} = {result}")
    return result

@tool
def power(base: float, exponent: float) -> float:
    """Raise a number to a power.
    
    Args:
        base: The base number
        exponent: The exponent
        
    Returns:
        base raised to the power of exponent
    """
    result = base ** exponent
    print(f"  🔢 {base}^{exponent} = {result}")
    return result

# Create math agent
math_tools = [add, multiply, power]
math_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)
math_llm_with_tools = math_llm.bind_tools(math_tools)

print("✅ Math tools created")

✅ Math tools created


In [14]:
# Build math agent (similar structure to weather agent)
class MathAgentState(TypedDict):
    messages: Annotated[list, add_messages]

def call_math_model(state: MathAgentState) -> dict:
    messages = state["messages"]
    response = math_llm_with_tools.invoke(messages)
    return {"messages": [response]}

def call_math_tools(state: MathAgentState) -> dict:
    messages = state["messages"]
    last_message = messages[-1]
    tool_calls = last_message.tool_calls
    
    tool_messages = []
    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"🧮 Executing: {tool_name}({tool_args})")
        
        tool = {t.name: t for t in math_tools}[tool_name]
        result = tool.invoke(tool_args)
        
        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )
    
    return {"messages": tool_messages}

def math_should_continue(state: MathAgentState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"

# Build graph
math_graph = StateGraph(MathAgentState)
math_graph.add_node("agent", call_math_model)
math_graph.add_node("tools", call_math_tools)
math_graph.add_edge(START, "agent")
math_graph.add_conditional_edges("agent", math_should_continue)
math_graph.add_edge("tools", "agent")

math_agent = math_graph.compile()

print("✅ Math agent ready!")

✅ Math agent ready!


In [15]:
# Test with complex calculation
query = "What is (5 + 3) multiplied by 2, then raised to the power of 2?"

print("=" * 60)
print(f"User: {query}")
print("=" * 60)

result = math_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

print("\n" + "=" * 60)
print(f"Agent: {result['messages'][-1].content}")
print("=" * 60)

# Show all messages
print("\n📝 Full conversation:")
for i, msg in enumerate(result["messages"]):
    print(f"{i+1}. {type(msg).__name__}: {msg.content if hasattr(msg, 'content') else 'Tool call'}")

User: What is (5 + 3) multiplied by 2, then raised to the power of 2?
🧮 Executing: add({'a': 5, 'b': 3})
  ➕ 5.0 + 3.0 = 8.0
🧮 Executing: multiply({'a': 2, 'b': 2})
  ✖️  2.0 × 2.0 = 4.0
🧮 Executing: multiply({'a': 8, 'b': 4})
  ✖️  8.0 × 4.0 = 32.0
🧮 Executing: power({'base': 32, 'exponent': 2})
  🔢 32.0^2.0 = 1024.0

Agent: The result of \((5 + 3) \times 2\) raised to the power of 2 is \(1024\).

📝 Full conversation:
1. HumanMessage: What is (5 + 3) multiplied by 2, then raised to the power of 2?
2. AIMessage: 
3. ToolMessage: 8.0
4. ToolMessage: 4.0
5. AIMessage: 
6. ToolMessage: 32.0
7. AIMessage: 
8. ToolMessage: 1024.0
9. AIMessage: The result of \((5 + 3) \times 2\) raised to the power of 2 is \(1024\).


## 6. Example 3: Multi-Tool Personal Assistant

Let's create a more realistic assistant that can handle multiple types of requests.

In [16]:
# Define diverse tools
@tool
def search_web(query: str) -> str:
    """Search the web for information (simulated).
    
    Args:
        query: The search query
        
    Returns:
        Search results as a string
    """
    # Simulated search results
    results = {
        "python": "Python is a high-level programming language known for its simplicity and readability.",
        "langgraph": "LangGraph is a framework for building stateful, multi-actor applications with LLMs.",
        "ai": "Artificial Intelligence (AI) is the simulation of human intelligence by machines."
    }
    
    for keyword, result in results.items():
        if keyword in query.lower():
            return f"Search results for '{query}': {result}"
    
    return f"No specific results found for '{query}'"

@tool
def get_current_time() -> str:
    """Get the current time.
    
    Returns:
        Current time as a string
    """
    from datetime import datetime
    return datetime.now().strftime("%I:%M %p on %B %d, %Y")

@tool
def calculate(expression: str) -> str:
    """Safely evaluate a mathematical expression.
    
    Args:
        expression: A mathematical expression (e.g., "2 + 2", "10 * 5")
        
    Returns:
        The result of the calculation
    """
    try:
        # Only allow basic math operations for safety
        allowed_chars = set("0123456789+-*/(). ")
        if not all(c in allowed_chars for c in expression):
            return "Error: Expression contains invalid characters"
        
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error calculating: {e}"

print("✅ Assistant tools created")

✅ Assistant tools created


In [17]:
# Create the assistant
assistant_tools = [search_web, get_current_time, calculate, get_weather]
assistant_llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)
assistant_llm_with_tools = assistant_llm.bind_tools(assistant_tools)

class AssistantState(TypedDict):
    messages: Annotated[list, add_messages]

def call_assistant_model(state: AssistantState) -> dict:
    response = assistant_llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def call_assistant_tools(state: AssistantState) -> dict:
    last_message = state["messages"][-1]
    tool_messages = []
    
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"\n🔧 Using tool: {tool_name}")
        print(f"   Args: {tool_args}")
        
        tool = {t.name: t for t in assistant_tools}[tool_name]
        result = tool.invoke(tool_args)
        
        print(f"   Result: {result}")
        
        tool_messages.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )
    
    return {"messages": tool_messages}

def assistant_should_continue(state: AssistantState) -> Literal["tools", "__end__"]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return "__end__"

# Build assistant graph
assistant_graph = StateGraph(AssistantState)
assistant_graph.add_node("agent", call_assistant_model)
assistant_graph.add_node("tools", call_assistant_tools)
assistant_graph.add_edge(START, "agent")
assistant_graph.add_conditional_edges("agent", assistant_should_continue)
assistant_graph.add_edge("tools", "agent")

assistant = assistant_graph.compile()

print("✅ Personal assistant ready!")

✅ Personal assistant ready!


In [18]:
# Test with various queries
test_queries = [
    "What time is it?",
    "What's the weather in Tokyo and what time is it there?",
    "Calculate 15 * 8 + 42",
    "Search for information about LangGraph"
]

for query in test_queries:
    print("\n" + "=" * 70)
    print(f"🧑 User: {query}")
    print("=" * 70)
    
    result = assistant.invoke({
        "messages": [HumanMessage(content=query)]
    })
    
    print(f"\n🤖 Assistant: {result['messages'][-1].content}")


🧑 User: What time is it?

🔧 Using tool: get_current_time
   Args: {}
   Result: 08:37 AM on May 09, 2026

🤖 Assistant: The current time is 08:37 AM on May 09, 2026.

🧑 User: What's the weather in Tokyo and what time is it there?

🔧 Using tool: get_weather
   Args: {'city': 'Tokyo'}
   Result: 🌍 Tokyo, Japan
🌡  Temperature : 19.7°C
💨 Wind speed  : 7.3 mph
☁  Condition   : Clear sky

🔧 Using tool: get_current_time
   Args: {}
   Result: 08:37 AM on May 09, 2026

🤖 Assistant: In Tokyo, the current weather is as follows:
- Temperature: 19.7°C
- Wind speed: 7.3 mph
- Condition: Clear sky

The current time in Tokyo is 08:37 AM on May 9, 2026.

🧑 User: Calculate 15 * 8 + 42

🔧 Using tool: calculate
   Args: {'expression': '15 * 8 + 42'}
   Result: 162

🤖 Assistant: The result of the calculation \(15 \times 8 + 42\) is 162.

🧑 User: Search for information about LangGraph

🔧 Using tool: search_web
   Args: {'query': 'LangGraph'}
   Result: Search results for 'LangGraph': LangGraph is a framewo

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware, ToolRetryMiddleware

## 🆕 Middleware: Built-in Retry & Content Moderation

LangGraph 1.1.9 (via langchain) introduces a middleware system for agents built with `create_agent`. Middleware wraps your agent with cross-cutting concerns like retry logic and content moderation.

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRetryMiddleware, PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return f"The weather in {city} is sunny and 22°C"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = create_agent(
    llm,
    tools=[get_weather],
    system_prompt="You are a helpful weather assistant.",
    middleware=[
        ModelRetryMiddleware(
            max_retries=3,
            initial_delay=1.0,
            backoff_factor=2.0,
            on_failure="error",
        ),
        # One instance per PII type
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        # `phone` isn't built-in — give it a regex detector
        PIIMiddleware(
            "phone",
            detector=r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b",
            strategy="redact",
            apply_to_input=True,
        ),
    ],
)

result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Paris?"}]})
print(result["messages"][-1].content)

The weather in Paris is sunny with a temperature of 22°C.


**Available middleware**:
- `RetryMiddleware(max_retries=3, backoff=2.0)` — exponential backoff retry on transient errors
- `ContentModerationMiddleware(check_input=True, check_output=True)` — OpenAI moderation API
- `SummarizationMiddleware()` — auto-summarize long conversation history
- Custom middleware via base class (covered in Notebook 07)

## 7. Hands-on Exercise: Enhanced Personal Assistant

Extend the personal assistant with additional capabilities!

### Your Task

Add these new tools:

1. **translate_text**: Translate text to another language (simulated)
2. **get_stock_price**: Get a stock price (simulated)
3. **set_reminder**: Create a reminder (just prints a message)

Then test with queries that use these tools.

In [26]:
# TODO: Implement these tools

@tool
def translate_text(text: str, target_language: str) -> str:
    """Translate text to another language (simulated).
    
    Args:
        text: The text to translate
        target_language: The target language (e.g., 'spanish', 'french')
        
    Returns:
        Translated text
    """
    # TODO: Implement simulated translation
    pass

# TODO: Add get_stock_price and set_reminder tools

# TODO: Create an enhanced assistant with all tools
# TODO: Test with queries like:
#   - "Translate 'Hello, how are you?' to Spanish"
#   - "What's the stock price of AAPL?"
#   - "Set a reminder to call John at 3pm"

<details>
<summary><b>Click for solution</b></summary>

```python
@tool
def translate_text(text: str, target_language: str) -> str:
    translations = {
        "spanish": {"hello": "hola", "how are you": "cómo estás"},
        "french": {"hello": "bonjour", "how are you": "comment allez-vous"}
    }
    lang = target_language.lower()
    if lang in translations:
        # Simple word replacement
        result = text.lower()
        for eng, trans in translations[lang].items():
            result = result.replace(eng, trans)
        return f"Translated to {target_language}: {result}"
    return f"Translation to {target_language} not available (simulated)"

@tool
def get_stock_price(symbol: str) -> str:
    stock_prices = {
        "AAPL": "$175.43",
        "GOOGL": "$142.87",
        "MSFT": "$378.91"
    }
    return stock_prices.get(symbol.upper(), f"Stock price for {symbol} not available (simulated)")

@tool
def set_reminder(task: str, time: str) -> str:
    return f"✅ Reminder set: '{task}' at {time}"
```
</details>

## 8. Key Takeaways

### Concepts Mastered

1. **Tool Calling**: LLMs can decide which tools to use
2. **@tool Decorator**: Converts Python functions into LangChain tools
3. **bind_tools()**: Attaches tools to an LLM
4. **add_messages Reducer**: Accumulates messages instead of replacing
5. **Agent Loop**: agent → tools → agent (until complete)
6. **Message Types**: HumanMessage, AIMessage, ToolMessage

### Best Practices

✅ **Write clear docstrings** - LLMs use them to understand tools  
✅ **Use type hints** - Helps LLMs understand parameters  
✅ **Keep tools focused** - Each tool should do one thing well  
✅ **Handle errors gracefully** - Tools should return error messages, not crash  
✅ **Test tools independently** - Before integrating with LLM  
✅ **Use GPT-4 for complex reasoning** - Better tool selection  
✅ **Set temperature=0** - More deterministic tool calling  

### Common Pitfalls

❌ Missing docstrings on tools  
❌ Tools that take too long to execute  
❌ Not handling tool execution errors  
❌ Forgetting to add tool results to messages  
❌ Creating infinite loops (missing END condition)  

### What's Next?

In **Notebook 04: State Persistence and Memory**, you'll learn:

- Adding conversation memory to agents
- Using checkpointers for persistence
- Multi-turn conversations
- Building stateful chatbots
- Thread-based conversation management

This will let you build agents that remember context across sessions!

## 9. Further Reading

- [LangChain Tools Documentation](https://python.langchain.com/docs/modules/agents/tools/)
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)
- [LangGraph Message Handling](https://langchain-ai.github.io/langgraph/concepts/low_level/#messages)
- [Building ReAct Agents](https://langchain-ai.github.io/langgraph/tutorials/introduction/#part-4-build-a-react-agent)

---

**Excellent progress!** You've built your first AI agents! Continue to [Notebook 04: State Persistence and Memory](04_state_persistence_and_memory.ipynb) to add memory to your agents!